In [1]:
import pandas as pd
import numpy as np
from darts import TimeSeries
from darts.dataprocessing.transformers import Scaler, MissingValuesFiller
from darts.models import TSMixerModel

# 1. 데이터 로드
df = pd.read_csv('../data/processed/smart_meter_data.csv')
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

# [Perplexity 제안 1] 통계 기반 클리핑 (음수 제거 및 상위 1% 이상치 억제)
# 유량(Flow_Instant)과 수압(Pressure) 모두에 적용
for col in ['Flow_Instant', 'Pressure']:
    upper_limit = df[col].quantile(0.99) # 상위 1% 값 계산
    df[col] = df[col].clip(lower=0, upper=upper_limit)

# [Perplexity 제안 2] 30분 단위 리샘플링 (피크 타임 보존)
df_resampled = df.set_index('Timestamp').resample('30T').mean().reset_index()

# 2. Darts 객체 생성 및 결측치 채우기
series = TimeSeries.from_dataframe(df_resampled, 'Timestamp', ['Flow_Instant', 'Pressure'])
filler = MissingValuesFiller() # 짧은 결측치는 기본 보간으로 처리
series_filled = filler.transform(series)

# 3. 스케일링 (0~1 정규화)
scaler = Scaler()
series_scaled = scaler.fit_transform(series_filled)

# 4. 데이터 분할 (30분 단위이므로 7일은 336개 시점 = 48개/일 * 7일)
# 학습 데이터는 2025-03-20 이전까지 사용
train, val = series_scaled.split_before(pd.Timestamp('2025-03-20'))

# 5. TSMixer 모델 설정
model = TSMixerModel(
    input_chunk_length=672,  # 과거 2주 (48 * 14)
    output_chunk_length=336, # 미래 1주일 (48 * 7)
    n_epochs=15,             # 30분 단위라 데이터가 늘었으므로 에포크 약간 상향
    batch_size=32,
    random_state=42
)

print("🚀 Perplexity의 제안이 반영된 모델 학습을 시작합니다 (30분 해상도)...")
model.fit(train)


ModuleNotFoundError: No module named 'pytorch_lightning'